In [1]:
%cd ../

/nas/zhangtianning.di/projects/unique_data_build


In [2]:
from tqdm.auto import tqdm

In [3]:
import redis

In [81]:
import redis
redis_host = "localhost"
redis_port = 6379
redis_password = ""
r = redis.StrictRedis(host=redis_host, port=redis_port, password=redis_password, decode_responses=True)

### read redis

In [84]:
alias_type = 'doi'  # The type of the alias you are looking for
alias_value = '10.1007/bf03046720'  # The alias for which you want to find the index

# Retrieve the index
index = get_index_by_alias(r,alias_type, alias_value)
print(f"The index for {alias_type} '{alias_value}' is: {index}")
a = get_alias_by_index(index)
print(a)

BusyLoadingError: Redis is loading the dataset in memory

### update crossref

In [7]:
from pathlib import Path
from tqdm.auto import tqdm
import gzip,json

resource_dir = "/nvme/zhangtianning.di/crossref/CrossrefData/"
resource_files= list(Path(resource_dir).glob("*.gz"))

filename = resource_files[0]
with gzip.open(filename, 'r') as f:
    json_data = json.load(f)

In [74]:
from tqdm.notebook import tqdm
filename = resource_files[0]
conflict_doi_that_should_same_but_splited = []
for filename in tqdm(resource_files, position=1, leave=False):
    Prefix_Code = str(filename).split('/')[-1].replace('.json.gz','')
    with gzip.open(filename, 'r') as f:
        json_data = json.load(f)
        for slot, t in tqdm(enumerate(json_data['items']),total=len(json_data['items']), position=2, leave=False):
            if 'alternative-id' in t and len(set(t['alternative-id']))>1:
                alternative_pool = [l.lower() for l in t['alternative-id']]
                primiry_doi      = t['DOI'].lower()
                alternative_doi  = [primiry_doi.replace(alternative_pool[0], sss) for sss in alternative_pool[1:]]
                alternative_doi  = [primiry_doi] + alternative_doi 
                doi_and_indexes  = [[doi,get_index_by_alias('doi', doi)] for doi in alternative_doi]
                the_valid_indexes= [index for doi, index in doi_and_indexes if index is not None]
                if len(the_valid_indexes)>1:
                    multirecord_doi = [doi for doi, index in doi_and_indexes if index is not None]
                    conflict_doi_that_should_same_but_splited.append(multirecord_doi)
                    continue
                if len(the_valid_indexes)==0:
                    the_record_index = f"CR.{Prefix_Code}.{slot}"
                else:
                    the_record_index = the_valid_indexes[0]
                    
                for doi, index in doi_and_indexes:
                    if index is not None:continue
                    print(slot,'doi', doi, the_record_index)
                    #add_alias_with_unique_name(r,'doi', doi, the_record_index)
            else:
                doi  = t['DOI']
                the_record_index = get_index_by_alias('doi', doi)
                if the_record_index is not None:
                    continue
                else:
                    the_record_index = f"CR.{Prefix_Code}.{slot}"
                    #add_alias_with_unique_name(r,'doi', doi, the_record_index)
            #for doi in alternative_doi:
                #print('doi', doi, the_record_index)
                #add_alias_with_unique_name(r,'doi', doi, the_record_index)
    break

  0%|          | 0/28701 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

2964 doi 10.18632/oncotarget.30515265 234767104
3857 doi 10.1145/3240765 139877237
3970 doi 10.1145/3240765.3272126 CR.9361.3970
3970 doi 10.1145/3240765 CR.9361.3970
4186 doi 10.1145/3240765 216603872
4494 doi 10.1145/3240765 224708238
4951 doi 10.18632/oncotarget.30515271 143889148


In [76]:
json_data['items'][4186 ]['alternative-id']

['10.1145/3240765.3240818', '10.1145/3240765']

In [70]:
get_index_by_alias('doi', doi)

In [67]:
conflict_doi_that_should_same_but_splited

[['10.1108/978-1-78743-327-420181003', '10.1108/9781787433274'],
 ['10.3920/978-90-8686-881-0_c09', '10.3920/978-90-8686-881-0'],
 ['10.3920/978-90-8686-881-0_c05', '10.3920/978-90-8686-881-0'],
 ['10.1002/9781119460435.refs', '10.1002/9781119460435'],
 ['10.3920/978-90-8686-881-0_fm', '10.3920/978-90-8686-881-0'],
 ['10.3920/978-90-8686-881-0_3', '10.3920/978-90-8686-881-0'],
 ['10.3920/978-90-8686-881-0_c13', '10.3920/978-90-8686-881-0'],
 ['10.1142/9789814528139', '10.1142/3914'],
 ['10.1108/978-1-78743-809-520181001', '10.1108/9781787438095'],
 ['10.1108/978-1-78743-809-520181006', '10.1108/9781787438095'],
 ['10.1108/978-1-78743-809-520181002', '10.1108/9781787438095'],
 ['10.1108/s0743-41542018000036c005',
  '10.1108/rhet',
  '10.1108/s0743-4154201836c'],
 ['10.1108/978-1-78754-531-120181007', '10.1108/9781787545311'],
 ['10.1145/3274247.3274516', '10.1145/3274247'],
 ['10.1108/978-1-78754-531-120181014', '10.1108/9781787545311'],
 ['10.1108/978-1-78754-531-120181001', '10.1108/9

In [54]:
resource_dir = "/nvme/zhangtianning.di/crossref/CrossrefData/"
resource_files= list(Path(resource_dir).glob("*.gz"))

In [57]:
resource_files = [str(t) for t in resource_files]

In [61]:
resource_dir = "/nvme/zhangtianning.di/crossref/crossref.filelist.json"
with open(resource_dir,'r') as f:
    resource_files= json.load(f)

In [52]:
import os
os.makedirs('/nvme/zhangtianning.di/crossref/CrossrefData_Conflict')

In [29]:
json_data['items'][2095]

{'URL': 'http://dx.doi.org/10.1108/978-1-78743-809-520181006',
 'resource': {'primary': {'URL': 'https://www.emerald.com/insight/content/doi/10.1108/978-1-78743-809-520181006/full/html'}},
 'member': '140',
 'score': 0.0,
 'created': {'date-parts': [[2018, 11, 7]],
  'date-time': '2018-11-07T06:45:51Z',
  'timestamp': 1541573151000},
 'license': [{'start': {'date-parts': [[2018, 11, 7]],
    'date-time': '2018-11-07T00:00:00Z',
    'timestamp': 1541548800000},
   'content-version': 'tdm',
   'delay-in-days': 0,
   'URL': 'http://www.emeraldinsight.com/page/tdm'},
  {'start': {'date-parts': [[2018, 11, 7]],
    'date-time': '2018-11-07T00:00:00Z',
    'timestamp': 1541548800000},
   'content-version': 'tdm',
   'delay-in-days': 0,
   'URL': 'http://www.emeraldinsight.com/page/tdm'}],
 'container-title': ['Platform Economics: Rhetoric and Reality in the ‘Sharing Economy’'],
 'issued': {'date-parts': [[2018, 11, 7]]},
 'prefix': '10.1108',
 'reference-count': 0,
 'indexed': {'date-parts':

In [28]:
json_data['items'][2060]

{'URL': 'http://dx.doi.org/10.1108/978-1-78743-809-520181001',
 'resource': {'primary': {'URL': 'https://www.emerald.com/insight/content/doi/10.1108/978-1-78743-809-520181001/full/html'}},
 'member': '140',
 'score': 0.0,
 'created': {'date-parts': [[2018, 11, 7]],
  'date-time': '2018-11-07T06:45:51Z',
  'timestamp': 1541573151000},
 'license': [{'start': {'date-parts': [[2018, 11, 7]],
    'date-time': '2018-11-07T00:00:00Z',
    'timestamp': 1541548800000},
   'content-version': 'tdm',
   'delay-in-days': 0,
   'URL': 'http://www.emeraldinsight.com/page/tdm'},
  {'start': {'date-parts': [[2018, 11, 7]],
    'date-time': '2018-11-07T00:00:00Z',
    'timestamp': 1541548800000},
   'content-version': 'tdm',
   'delay-in-days': 0,
   'URL': 'http://www.emeraldinsight.com/page/tdm'}],
 'container-title': ['Platform Economics: Rhetoric and Reality in the ‘Sharing Economy’'],
 'issued': {'date-parts': [[2018, 11, 7]]},
 'prefix': '10.1108',
 'reference-count': 0,
 'indexed': {'date-parts':

### update unarxive

In [349]:
def sort_section_numbers(section_numbers):
    def split_section_number(section_number):
        return list(map(int, section_number.split('.')))

    sorted_numbers = sorted(section_numbers, key=split_section_number)
    return sorted_numbers

section_numbers = ['2.2.1', '1.1', '2.2', '1', '1.2', '2', '1.1.1']
sorted_numbers = sort_section_numbers(section_numbers)
print(sorted_numbers)

['1', '1.1', '1.1.1', '1.2', '2', '2.2', '2.2.1']


In [344]:
done_file_names = [x.name for x in Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl')]
all_raw_files   = list(Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl'))
all_raw_files = [str(t) for t  in all_raw_files]
with open("/nvme/zhangtianning.di/datasets/unarxive/whole_filelist.json",'w') as f:
    json.dump(all_raw_files, f)

In [345]:
done_file_names = [x.name for x in Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl')]

In [346]:
all_raw_files   = list(Path('/nvme/zhangtianning.di/datasets/unarxive/').glob('**/*.jsonl'))

In [391]:
from tqdm.auto import tqdm

#### convert unarxive format into uparxive format

In [100]:
from clean_unarXive_data import *

In [104]:
import tiktoken
encoding = tiktoken.get_encoding("cl100k_base")

In [354]:
##############
metadata_i    = data['metadata']
body_text_i   = data['body_text']
bib_entries_i = data['bib_entries']
ref_entries_i = data['ref_entries']
paper_id = data['paper_id']

# get a dict for format citation/reference data
insert_data = {}
type_list = []

# for formula, figure, table
figure_ref  = {}
table_ref   = {}
equation_ref= {}
figures_metadata ={}
tables_metadata  ={}

paper_unique_id  = identify_string_type(metadata_i['id'])
for _id, item in ref_entries_i.items():
    type_list.append(item['type'])
    key = "{}:{}".format(item['type'], _id)
    if item['type'] == 'formula':
        value = """${}$""".format(item['latex'])
    elif item['type'] == 'table':
        value = f"""[Table.{len(tables_metadata)} of {paper_unique_id}]"""
        tables_metadata[_id] = item
        table_ref[_id] =[ 'Table', len(tables_metadata)]
    elif item['type'] == 'figure':
        value = f"""[Figure.{len(figures_metadata)} of {paper_unique_id}]"""
        figures_metadata[_id] = item
        figure_ref[_id]  = ['Figure',len(figures_metadata)]
    else:
        print('='*20)
        print(item)
        print('='*20)
        value = ""
    insert_data[key] = value

# remove duplicate
type_list = list(set(type_list)) 

# for citation
type_list.append('cite')

for i,(_id, item) in enumerate(bib_entries_i.items()):
    key = "{}:{}".format('cite', _id)
    value = get_name(item)
    value = f"[Ref.{i} of {paper_unique_id}]" if value is None else value
    insert_data[key] = value

all_text = []
Reference= []
ReferenceQ= False
acknowledgement = None
for paragraph_id, paragraph in enumerate(body_text_i):
    #print(paragraph.keys())
    text = paragraph['text']

    start_string = text.strip()
    if start_string.startswith('Acknowledgement'):
        acknowledgement = new_text
        continue
    if start_string.startswith('REFERENCES') or start_string.startswith('Reference'):
        ReferenceQ=True
        print(f"fail at paragraph {paragraph_id}")
        print(paragraph)
        with open('fail.json','w') as f:
            json.dump(data, f)
        raise

    new_text = format_text_with_values(text, insert_data, type_list,paper_unique_id)
    if len(new_text)==0: continue
    if not ReferenceQ:   
        paragraph['format_text'] = new_text
    else:
        Reference.append(new_text)

sections = []
structued_paragraph = {}
for flatten_paragraph in body_text_i:
    section_num  = flatten_paragraph['sec_number']
    section_num  = section_num.strip()
    if section_num not in structued_paragraph:
        structued_paragraph[section_num] = {'section_content':[]}

    section_name = flatten_paragraph['section']
    section_name = better_latex_sentense_string(section_name)

    if section_name in structued_paragraph[section_num]:
        assert section_name == structued_paragraph[section_num], f"why we get two different section name for [{section_name}] and ]"
    structued_paragraph[section_num]['section_name'] = section_name
    if 'format_text' not in flatten_paragraph:
        #print(flatten_paragraph)
        continue
    new_text = flatten_paragraph['format_text']
    all_text = structued_paragraph[section_num]['section_content']

    if all_text and new_text.strip() and (new_text.strip()[0].islower() or new_text.strip().startswith('Proof')):
        all_text[-1]+= ' ' + new_text.strip()
    elif all_text and all_text[-1].strip() and (all_text[-1].strip()[-1]=="$" or all_text[-1].strip()[-1]==":") and len(all_text[-1].split())<128:
        all_text[-1]+= ' ' + new_text
    else:
        all_text.append(new_text.replace('\n'," "))
    structued_paragraph[section_num]['section_content'] = all_text

In [383]:
!pip install roman

Looking in indexes: https://pkg.pjlab.org.cn/repository/pypi-proxy/simple/


In [386]:
import re
import roman  # You'll need to install the 'roman' package for this

# First, define a helper function to determine if a string is an integer
def is_integer(s):
    try:
        int(s)
        return True
    except ValueError:
        return False

# Define a helper function to determine if a string is a Roman numeral
def is_roman_numeral(s):
    try:
        roman.fromRoman(s)
        return True
    except roman.InvalidRomanNumeralError:
        return False

# Define a function to convert section parts to a sortable key
def section_key(section):
    # Split the section into parts (e.g., '2.3.1' -> ['2', '3', '1'])
    parts = re.split(r'\.', section)
    key = []
    
    for part in parts:
        if is_integer(part):  # Integer case
            key.append(('int', int(part)))
        elif is_roman_numeral(part):  # Roman numeral case
            key.append(('roman', roman.fromRoman(part)))
        elif part.isalpha():  # Letter case
            key.append(('letter', part.upper()))  # treat 'a' and 'A' equally
        else:
            raise ValueError(f"Unknown section format: {part}")
    
    return key

# Now you can sort a list of section numbers
def sort_sections(sections):
    return sorted(sections, key=section_key)

# Example```python
import re

# Helper functions to identify and convert section numbering formats
def int_or_roman_to_int(value):
    # Converts integers and roman numerals to integers for comparison
    try:
        return int(value)
    except ValueError:
        # Assume roman numeral since it's not an integer
        return roman_to_int(value)

def roman_to_int(value):
    # Converts roman numerals to integers
    roman_numerals = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    int_val = 0
    for i in range(len(value)):
        if i > 0 and roman_numerals[value[i]] > roman_numerals[value[i - 1]]:
            int_val += roman_numerals[value[i]] - 2 * roman_numerals[value[i - 1]]
        else:
            int_val += roman_numerals[value[i]]
    return int_val

def section_key(section):
    # Converts a section into a tuple of keys for sorting
    parts = re.split('\.|-', section)  # Split on . or -
    key_parts = []
    
    for part in parts:
        if part.isdigit():  # Integer part
            key_parts.append(int(part))
        elif part.isalpha():  # Letter part
            # Convert letters to integers based on alphabetical position
            # Make sure to handle lower and uppercase letters identically
            key_parts.append(ord(part.upper()) - ord('A') + 1)
        elif re.match(r'^[IVXLCDM]+$', part):  # Roman numeral part
            key_parts.append(roman_to_int(part))
        else:
            # Handle invalid section formats
            raise ValueError(f"Invalid section format: {part}")
    
    return tuple(key_parts)

# Sort a given list of section numbers
def sort_section_numbers(section_numbers):
    return sorted(section_numbers, key=section_key)

# Example usage
section_numbers = ["1", "2", "3.1", "3.2", "10", "A", "B", "C", "I", "II", "III", "X", "V"]
sorted_sections = sort_section_numbers(section_numbers)
print(sorted_sections)

TypeError: ord() expected a character, but string of length 2 found

In [385]:
import re

def sort_section_numbers(section_numbers):
    def split_section_number(section_number):
        parts = re.split(r'(\d+)', section_number)
        return [int(part) if part.isdigit() else part for part in parts]

    sorted_numbers = sorted(section_numbers, key=split_section_number)
    return sorted_numbers

section_numbers = ['2.2.1', '1.1', '2.2', '1', '1.2', '2', '1.1.1', '2.2.A', '2.B', '1.1.C']
sorted_numbers = sort_section_numbers(section_numbers)
print(sorted_numbers)

['1', '1.1', '1.1.1', '1.1.C', '1.2', '2', '2.2', '2.2.1', '2.2.A', '2.B']


In [379]:
split_by_indentation(structued_paragraph['3']['section_content'])

["Before deriving Eq.(REF ), it will be helpful to summarize the procedure for obtaining the standard Bell Inequalities in order to point out similarities to the CHSH-Eberhard experiment. The reader is referred to SR or Sakurai (See [DOI:10.1017/9781108587280]) for additional details; see also the Appendix. Like its successor, Bell's theorem is valid for local hidden-variable theories, which involve only classical probabilities. In a typical derivation such as Sakurai's one assumes that spin measurements may be made along any of three axes, a, b and c. A system of decaying atoms emits $N$ particles of which a certain fraction are taken to be, say, of the type (a+, b+, c+) $\\equiv (+++)$ , which designates spin up along all three axes. To ensure zero total angular momentum, each emitted particle of type (+++) must be paired with one of type ($ - - - $ ). There are eight such spin combinations in all, as listed in Table 1. The probability that (+++) is emitted (and in the case of hidden

In [353]:
sections = []
appendix = []
section_num_keys = list(structued_paragraph.keys())
section_num_keys = sort_section_numbers(section_num_keys)

for section_num in section_num_keys:
    section_pool  = structued_paragraph[section_num]
    section_name   = section_pool['section_name']
    appendex_mode = False
    if section_name.lower() in ['appendix','supplementary'] or '-' in section_num:
        appendex_mode = True

    if not section_name: 
        if not appendex_mode:
            section_name = f'Section {section_num}'
        else:
            section_name = f"Appendix {section_num.replace('-','')}"
    now_section = {'section_title':section_name}|{'section_content':split_by_indentation(section_pool['section_content']),
                                                  'section_num':section_num}
    if not appendex_mode:
        sections.append(now_section)
    else:
        appendix.append(now_section)

In [142]:
from dataclasses import dataclass

In [ ]:
from typing import List

In [160]:
from python_script.reference_reterive.Reference import flatten_dict
from xml_to_dense_text import better_latex_sentense_string
class BaseElement:
    def to_dict(self):
        return vars(self)
    
    def to_flatten_dict(self):
        out = flatten_dict(self.to_dict())
        
        return out
    

@dataclass
class Section(BaseElement):
    section_title: str = None
    section_content: List[str] = None

@dataclass
class PaperMetadata(BaseElement):
    figures_metadata: dict = None
    tables_metadata: dict = None
    bibitem_ref_metadata: dict = None

@dataclass
class Paper(BaseElement):
    paper_id: str = None
    abstract: str = None
    acknowledge: str = None
    sections: List[Section] = None
    appendix: List[Section] = None
    metadata: PaperMetadata = None
    whole_ref_to_labels: Dict[str, list] = None
    missing_citation_labels: dict = None
    def to_dict(self):
        return vars(self)
    
    def to_flatten_dict(self):
        out = flatten_dict(self.to_dict())
        return out
    @staticmethod
    def get_paper_id(pool):
        return pool.get('paper_id', None)
    
    
    
    @staticmethod
    def abstract(pool):
        abstract =  pool.get('abstract', None)
        if isinstance(abstract, dict):
            abstract = abstract['text']
        abstract = better_latex_sentense_string(abstract)
        return abstract

In [ ]:

#os.makedirs(output_dir, exist_ok=True)

#Content_Paht = os.path.join(output_dir, f'{_paper_id}.retrieved.json') if reterive_result_mode else os.path.join(output_dir, f'{_paper_id}.json')
# with open(Content_Paht, 'w') as f:json.dump(output_dict, f, indent=2)
# if not reterive_result_mode:
#     keys  = list(bibitem_ref_metadata.keys())
#     citation_string = [bibitem_ref_metadata[key] for key in keys]
#     with open(os.path.join(output_dir, f'reference.keys'), 'w') as f:
#         for key in keys:f.write(key+'\n')
#     with open(os.path.join(output_dir, f'reference.txt'), 'w') as f:
#         for string in citation_string:f.write(string+'\n')
#     with open(os.path.join(output_dir, f'bibitem_ref_metadata_not_in_context.json'), 'w') as f:
#         json.dump(bibitem_ref_metadata_not_in_context, f, indent=2)
#     with open(os.path.join(output_dir, f'note_ref_metadata_not_in_context.json'), 'w') as f:
#         json.dump(note_ref_metadata_not_in_context, f, indent=2)

In [110]:
for key, val in data['bib_entries'].items():
    print(UniqueID.from_dict(val['ids']))
    break

Paper:
doi |-> 10.1119/1.13804
openalex |-> w1987630316
semopenalex |-> w1987630316


In [105]:
from python_script.reference_reterive.Reference import UniqueID

In [111]:
paper = UniqueID.from_dict(val['ids'])

In [116]:
with open("/nvme/zhangtianning.di/openalex/paper_identity.jsonl",'r') as f:
    for line in f:
        identity_set = json.loads(line)
        break

In [117]:
identity_set

{'openalex': 'W4312243914', 'doi': '10.3030/750533'}

In [115]:
paper.to_dict()

{'doi': '10.1119/1.13804',
 'openalex': 'w1987630316',
 'semopenalex': 'w1987630316'}

In [119]:
all_raw_files[0]

'/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0004_001.jsonl'

In [123]:
paper.to_dict()

{'doi': '10.1119/1.13804',
 'openalex': 'w1987630316',
 'semopenalex': 'w1987630316'}

In [138]:
conflict_doi_that_should_same_but_splited_dir

'/nvme/zhangtianning.di/datasets/unarxive_Conflict/00'

In [137]:
filename = all_raw_files[0]
Prefix_Code = str(filename).split('/')[-1].replace('arXiv_src_','').replace('jsonl','')
conflict_doi_that_should_same_but_splited = []
conflict_doi_that_should_same_but_splited_path = str(filename).replace('unarxive','unarxive_Conflict').replace('.jsonl','.json')
conflict_doi_that_should_same_but_splited_dir  = os.path.dirname(conflict_doi_that_should_same_but_splited_path)
os.makedirs(conflict_doi_that_should_same_but_splited_dir)

In [156]:

#if os.path.exists(conflict_doi_that_should_same_but_splited_path):continue
with open(filename, 'r') as f:
    length = len(f.readlines())
with open(filename, 'r') as f:
    for slot, line in tqdm(enumerate(f),total=length, position=2, leave=False):
        data = json.loads(line)
        if 'bib_entries' in data:
            for key, val in data['bib_entries'].items():
                if 'ids' not in val:continue
                paper = UniqueID.from_dict(val['ids'])
                if paper.is_nan():continue
                doi_and_indexes= []
                for alias_type,alias_value in paper.to_dict().items():
                    if alias_type == 'semopenalex':continue
                    if 'openalex' in alias_type:alias_value = alias_value.replace('w','W')
                    doi_and_indexes.append([alias_type, alias_value, get_index_by_alias(r,alias_type, alias_value)])
                the_valid_indexes= list(set([index for alias_type, alias_value, index in doi_and_indexes if index is not None]))
                if len(the_valid_indexes)>1:
                    multirecord_doi = [[alias_type, alias_value] for alias_type, alias_value, index in doi_and_indexes if index is not None]
                    conflict_doi_that_should_same_but_splited.append(multirecord_doi)
                    continue
                if len(the_valid_indexes)==0:
                    the_record_index = f"UA.{Prefix_Code}.{slot}"
                    for alias_type, alias_value, index in doi_and_indexes:
                        unique_name = format_alias(alias_type, alias_value)
                        add_alias_with_unique_name(r,alias_type, alias_value, the_record_index)
                else:
                    the_record_index = the_valid_indexes[0]
                    old_set = get_alias_by_index(r,the_record_index)
                    for alias_type, alias_value, index in doi_and_indexes:
                        unique_name = format_alias(alias_type, alias_value)
                        if unique_name in old_set:continue
                        add_alias_with_unique_name(r,alias_type, alias_value, the_record_index)
                

  0%|          | 0/1876 [00:00<?, ?it/s]

pubmed:10045028
pubmed:10038115
pubmed:10034381
pubmed:10056476
arxiv:hep-th/9306039
pubmed:10057575
pubmed:10058410
pubmed:10059978
pubmed:10059979
arxiv:quant-ph/9811018
pubmed:10058408
arxiv:quant-ph/9503017
pubmed:10058409
pubmed:9912655
pubmed:9912645
arxiv:quant-ph/9503016
arxiv:quant-ph/9902071
pubmed:10059671
arxiv:quant-ph/9604034v1
pubmed:9902079
pubmed:9899572
pubmed:9960745
pubmed:9963242
openalex:W2502808615
pubmed:9963241
pubmed:10044956
pubmed:10060692
arxiv:quant-ph/9602004
pubmed:10061534
arxiv:quant-ph/9511027
openalex:W3005571154
pubmed:9903985
pubmed:9913432
pubmed:10045774
pubmed:9897735
pubmed:10058872
openalex:W2028406173
arxiv:quant-ph/9806029
pubmed:10046665
arxiv:cs/9910010
arxiv:quant-ph/9704026
arxiv:quant-ph/9711065
arxiv:quant-ph/9605044
arxiv:quant-ph/9710013
pubmed:9913930
arxiv:quant-ph/9604024
pubmed:9910380
arxiv:hep-th/9305062
arxiv:quant-ph/9711021
arxiv:quant-ph/9711049
arxiv:quant-ph/9810082
pubmed:9913410
openalex:W3117619190
pubmed:9902211
arxiv

arxiv:quant-ph/9706033
pubmed:10059978
pubmed:10060523
arxiv:quant-ph/9904096
pubmed:10058410
arxiv:quant-ph/9810039
arxiv:quant-ph/9810040
arxiv:quant-ph/9810087
arxiv:quant-ph/9806021
arxiv:quant-ph/9903044
openalex:W2057883617
pubmed:9909547
pubmed:9910869
arxiv:quant-ph/9801025
openalex:W2031016911
pubmed:10060907
arxiv:quant-ph/9511007
openalex:W2167050139
openalex:W2799150811
pubmed:10096793
pubmed:17837304
pubmed:10043760
pubmed:11299042
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/31338
openalex:W1785119608
pubmed:9910979
pubmed:19559947
pubmed:3797887
openalex:W1979700341
pubmed:10060366
pubmed:10062233
arxiv:cond-mat/9904399
pubmed:9908238
pubmed:9910453
arxiv:quant-ph/9801076
arxiv:hep-th/9712038
arxiv:quant-ph/0406158
arxiv:quant-ph/9903098
arxiv:hep-th/9210067
pubmed:9959337
arxiv:physics/9805038
arxiv:quant-ph/9906062
pubmed:9908632
pubmed:9913077
arxiv:quant-ph/9711039
arxiv:quant-ph/9803031
pubmed:9913432
pubmed:9913637
pubmed:10063159
pubmed:9909314
arxiv:quant-ph/980

openalex:W1560060148
openalex:W2010154591
pubmed:7701342
openalex:W1936774573
pubmed:4559589
pubmed:9441817
arxiv:physics/9806046
pubmed:11969649
pubmed:10039074
arxiv:cond-mat/9803340
arxiv:chao-dyn/9705007v1
pubmed:9963660
pubmed:9898277
openalex:W3101415193
arxiv:nlin/0004022
arxiv:hep-th/9712232
openalex:W2008891504
arxiv:solv-int/9706006v1
arxiv:solv-int/9808008
openalex:W2045394134
pubmed:9898882
pubmed:9904456
pubmed:10786783
pubmed:9964954
pubmed:10046278
arxiv:math/9912143
openalex:W1691157804
arxiv:math/9909010
arxiv:math-ph/9907025v1
arxiv:math/9907165
openalex:W2062565740
arxiv:math/9201261v1
arxiv:solv-int/9706002
openalex:W1512385898
openalex:W2083884937
openalex:W3100333039
arxiv:math/9903134
arxiv:math/9906120
arxiv:math/9907127v3
arxiv:solv-int/9810004
arxiv:math/9912025
arxiv:math/9904042
arxiv:hep-th/9306042
doi:10.1142/9789814531450
openalex:W2765595164
openalex:W2083960709
pubmed:10061448
pubmed:10058476
arxiv:adap-org/9501001
arxiv:cond-mat/9711299
arxiv:patt-sol/

arxiv:hep-th/9803091
pubmed:9959053
pubmed:10017884
arxiv:gr-qc/9403031
arxiv:hep-th/0503203
pubmed:10015165
arxiv:1204.4658
arxiv:hep-th/9708046
pubmed:10033117
pubmed:10031711
arxiv:hep-th/9705035
arxiv:hep-th/9801073
arxiv:gr-qc/9806065
arxiv:hep-th/9906006
arxiv:hep-th/9610252
arxiv:hep-th/9711027
pubmed:9956885
pubmed:10016333
arxiv:gr-qc/9306015
pubmed:10018435
arxiv:hep-th/9501066
arxiv:hep-th/9509127
pubmed:15222900
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/459214
pubmed:9958553
pubmed:10059734
arxiv:hep-th/9504083
pubmed:15222900
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/459214
pubmed:16635264
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/1479323
pubmed:15348269
openalex:W3098671359
arxiv:hep-th/9906194
arxiv:hep-th/9909121
pubmed:15222900
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/459214
pubmed:10411109
pubmed:15892874
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/1175958
pubmed:15222900
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/459214
arxiv:hep-th/9803002
pubm

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7fae9d97e230>>
Traceback (most recent call last):
  File "/home/zhangtianning.di/anaconda3/envs/unarxive/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 770, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


arxiv:hep-ph/9905221
arxiv:hep-th/9211021
openalex:W2027761034
openalex:W2006574587
arxiv:hep-th/9501055
arxiv:hep-th/9706179
pubmed:10020421
arxiv:hep-th/9509074
arxiv:gr-qc/9903063
arxiv:hep-th/9907067
pubmed:10018138
arxiv:hep-th/9406216
arxiv:gr-qc/9906085
openalex:W1586267306
arxiv:gr-qc/9511066
arxiv:gr-qc/9506015
arxiv:gr-qc/9607062
arxiv:gr-qc/9805085
arxiv:gr-qc/9803078
pubmed:10041934
arxiv:hep-th/9906064
arxiv:hep-ph/9905221
pubmed:15222900
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/459214
arxiv:hep-th/0003117
arxiv:hep-ph/0003129
arxiv:hep-ph/9908530
arxiv:hep-ph/9909262
arxiv:hep-th/0002140
arxiv:hep-th/0003100
pubmed:10038431
pubmed:8378254
pubmed:15222900
pmc:https://www.ncbi.nlm.nih.gov/pmc/articles/459214
pubmed:10966653
openalex:W1560378513
arxiv:hep-th/9807069
pubmed:10057022
arxiv:hep-th/9405005
pubmed:10053982
arxiv:hep-th/9303012
pubmed:10055907
arxiv:hep-th/9401059
arxiv:hep-th/9912207
pubmed:9736684
openalex:W4206688401
arxiv:hep-th/9704142v1
openalex:W203803


KeyboardInterrupt



In [ ]:
keys

In [ ]:
with open(os.path.join(output_dir, f'reference.keys'), 'w') as f:
    for key in keys:f.write(key+'\n')
with open(os.path.join(output_dir, f'reference.txt'), 'w') as f:
    for string in citation_string:f.write(string+'\n')

In [149]:
conflict_doi_that_should_same_but_splited

[[['doi', '10.1007/b100336'], ['openalex', 'W1534848731']],
 [['doi', '10.1007/b100336'], ['openalex', 'W1534848731']],
 [['doi', '10.1007/b100336'], ['openalex', 'W1534848731']],
 [['doi', '10.1038/155772b0'], ['openalex', 'W3189468045']],
 [['doi', '10.1103/physrevb.37.3575'], ['openalex', 'W2007433768']],
 [['doi', '10.1126/science'], ['openalex', 'W4214911971']]]

In [131]:
get_index_by_alias(r,'doi', '10.1119/1.13804')

'136446769'

In [132]:
get_index_by_alias(r,'openalex', 'W1987630316')

'136446769'

In [133]:
conflict_doi_that_should_same_but_splited

[]

In [83]:
with tqdm.tqdm(range(start, end), desc="Main", ncols=100, ascii=True) as pbar:
    for i in pbar:
        file_name = all_files[i]
        convert_a_file(file_name)
        pbar.update(1)


[PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0004_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0002_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0001_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0003_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0005_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0006_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0007_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0009_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0008_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0010_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0011_001.jsonl'),
 PosixPath('/nvme/zhangtianning.di/datasets/unarxive/00/arXiv_src_0012_001.j

### update arxive-office

In [3]:
from tqdm.auto import tqdm
i=0
with open('/nvme/zhangtianning.di/datasets/whole_arxiv_data/arxiv-metadata-oai-snapshot.json','r') as f:
    for line in tqdm(f):
        i+=1

0it [00:00, ?it/s]

In [8]:
end_index = 10 
start_index=0

In [10]:
resource_dir = '/nvme/zhangtianning.di/datasets/whole_arxiv_data/arxiv-metadata-oai-snapshot.json'


In [21]:
import json,os
conflict_doi_that_should_same_but_splited=[]

In [6]:
from python_script.reference_reterive.Reference import UniqueID

In [5]:
from python_script.build_redis_database.utils import *

In [ ]:
"doi", "10.1109/5992.774844"], ["arxiv", "cond-mat/9809122"

In [79]:
get_index_by_alias(r,"doi", "10.1103/PhysRevLett.88.013601")

'120886759'

In [78]:
redis = r

In [80]:
get_alias_by_index(redis,'120886759')

{'arxiv:quant-ph/0111112',
 'arxiv_id:quant-ph/0111112',
 'doi:10.1103/PhysRevLett.88.013601',
 'doi:10.1103/physrevlett.88.013601',
 'mag:2067430275',
 'openalex:W2067430275',
 'pmid:11800943',
 'pubmed:11800943'}

In [8]:
paper = UniqueID.from_dict({'arxiv':"cond-mat/9809122", 'doi':"10.1109/5992.774844"})
doi_and_indexes= []

In [10]:
doi_and_indexes= []
for alias_type,alias_value in paper.to_dict().items():
    doi_and_indexes.append([alias_type, alias_value, get_index_by_alias(r,alias_type, alias_value)])
the_valid_indexes= list(set([index for alias_type, alias_value, index in doi_and_indexes if index is not None]))


In [28]:
import json
with open("/nvme/zhangtianning.di/datasets/whole_arxiv_data/arxiv-metadata_conflict/0_2.json",'r') as f:
    arxiv_conflict_pool = json.load(f)

In [33]:
arxivpool = set([arxiv[1] for doi, arxiv in arxiv_conflict_pool])

In [62]:
from pathlib import Path

In [64]:
from tqdm.auto import tqdm

In [68]:
Files

<generator object Path.glob at 0x7efcd0686a40>

In [70]:
len(ListP)

1148

In [72]:
Files = list(Path('/nvme/zhangtianning.di/crossref/updateitem/unused_doi').glob('*.escape_by_too_less_information'))
ListP = []
for File in tqdm(Files):
    with open(File,'r' ) as f:
        for line in f:
            ListP.append(line.strip())
with open('/nvme/zhangtianning.di/crossref/updateitem/escape_by_too_less_information','w' ) as f:
    for line in ListP:
        f.write(line+'\n')

  0%|          | 0/28701 [00:00<?, ?it/s]

In [73]:
Files = list(Path('/nvme/zhangtianning.di/crossref/updateitem/unused_doi').glob('*.escape_by_no_unique_id'))
ListP = []
for File in tqdm(Files):
    with open(File,'r' ) as f:
        for line in f:
            ListP.append(line.strip())
with open('/nvme/zhangtianning.di/crossref/updateitem/escape_by_no_unique_id','w' ) as f:
    for line in ListP:
        f.write(line+'\n')

  0%|          | 0/28701 [00:00<?, ?it/s]

In [63]:
AnalysisFiles = Path('/nvme/zhangtianning.di/crossref/updateitem/unused_doi').glob('*.Analysis.json')
Analysis={}
for AnalysisFile in tqdm(AnalysisFiles):
    with open(AnalysisFile, 'r') as f:
        AnalysisNow = json.load(f)
    for key, val in AnalysisNow.items():
        if key not in Analysis:
            Analysis[key] = val
        else:
            Analysis[key] += val

In [66]:
Analysis

{'update': 89012700, 'new_add': 26504477, 'crossref_normal_skip': 4111124}

In [65]:
with open('/nvme/zhangtianning.di/crossref/updateitem/Analysis.json','w') as f:
    json.dump(Analysis, f)

### semnitic scholar

In [28]:
from pathlib import Path

In [50]:
all_paths = list(Path("/nvme/zhangtianning.di/semantic_scholar/split/").glob('20231201*'))
all_paths = [str(t) for t in all_paths]

In [73]:
with open("/nvme/zhangtianning.di/semantic_scholar/all_filepath.json", 'w') as f:
    json.dump(all_paths, f)

In [29]:
with open("/nvme/zhangtianning.di/semantic_scholar/vene_alias_map.json", 'r') as f:
    vene_name_to_alias = json.load(f)

In [ ]:
_id_names        = ['DOI', 'ArXiv', 'DBLP', 'ACL', 'PubMed', 'PubMedCentral']
short_id_names   = ['DOI', 'ArXiv', 'DBLP', 'ACL', 'PubMed', 'PubMedCentral']
exteral_id_names = ['externalids.{}'.format(_id_name) for _id_name in _id_names]

In [31]:
with open(all_paths[0], mode="r") as f:
    for line in f:
        data = json.loads(line)
        break

In [51]:
filename = all_paths[0]

In [52]:
filename

'/nvme/zhangtianning.di/semantic_scholar/split/20231201_080654_00026_4iy4k_07c2399c-a2a7-4045-9616-864e9d817ebc.part_aa'

In [53]:
num = 1
part = filename.split('/')[-1].split('.')[-1]

In [68]:
@dataclass
class UniqueID:
    doi: Optional[str] = None
    openalex: Optional[str] = None
    pubmed: Optional[str] = None
    semopenalex: Optional[str] = None
    arxiv: Optional[str] = None
    pmc: Optional[str] = None
    dblp: Optional[str] = None
    mag: Optional[str] = None
    pii: Optional[str] = None
    acl: Optional[str] = None
        
    def is_same(a, b):
        pool_a = a.to_dict()
        pool_b = b.to_dict()
        for key, val_a in pool_a.items():
            if key in pool_b:
                val_b = pool_b[key]
            else:
                nkey = UniqueID.unique_key(key)
                if nkey in pool_b:
                    val_b = pool_b[nkey]
                else:
                    continue
            if val_a == val_b:return True
        return  False

    def is_nan(self):
        return all([v is None for v in self.__dict__.values()])
    
    

    @staticmethod
    def unique_key(key):
        key = key.lower().replace('_',"").strip()
        if key.endswith('id'): key = key[:-2]
        if key in ['pubmed', 'pm']:
            return 'pubmed'
        elif key in ['pubmedcentral', 'pmc']:
            return 'pmc'
        return key
    
    @staticmethod
    def from_dict(d:Dict[str,str]):
        new_ids = {}
        for key in ["DOI", "ArXiv", "DBLP",'PubMed','PMID','PMC','MagID','Pii','Pmcid','ArXivId','PMCID']:
            _id = d.get(f'externalids.{key}',None)
            if _id is not None and len(_id)>0:
                new_ids[UniqueID.unique_key(k)] = _id.lower()

        for k,v in d.items():
            if v is None:continue
            if len(v) ==0:continue
            v = v.lower()
            k = UniqueID.unique_key(k)
            if k == 'openalex':
                v = v.replace('https://openalex.org/','')
            elif k == 'semopenalex':
                v = v.replace('https://semopenalex.org/work/','')
            elif k == 'pubmed':
                v = v.replace('https://pubmed.ncbi.nlm.nih.gov/','')
            else:
                v = v
            
            new_ids[k] = v
        return UniqueID(**new_ids)
    
    def __repr__(self):
        info = "\n".join([f"{key} |-> {val}" for key, val in self.to_dict().items() if val is not None])
        return f"Paper:\n{info}"   

    def to_dict(self):
        return {k:v for k,v in vars(self).items() if v is not None}


In [71]:
part = os.path.split(filename)[-1].split('.')[-1]

In [72]:
part

'part_aa'

In [70]:
Prefix_Code = f'SS{num}.{part}'
conflict_doi_that_should_same_but_splited = []
conflict_doi_that_should_same_but_splited_dir  = os.path.join(os.path.dirname(os.path.dirname(filename)),'semantic_scholar_conflict')
conflict_doi_that_should_same_but_splited_path = os.path.join(conflict_doi_that_should_same_but_splited_dir, f"{Prefix_Code}.json")
os.makedirs(conflict_doi_that_should_same_but_splited_dir, exist_ok=True)
with open(filename, 'r') as f:
    length = len(f.readlines())
with open(filename, 'r') as f:
    for slot, line in tqdm(enumerate(f),total=length, position=2, leave=False):
        data = json.loads(line)
        ids  = data['externalids']
        ids.pop('CorpusId')
        paper = UniqueID.from_dict(ids)
        if paper.is_nan():continue
        doi_and_indexes= []
        for alias_type,alias_value in paper.to_dict().items():
            if alias_type == 'semopenalex':continue
            if 'openalex' in alias_type:alias_value = alias_value.replace('w','W')
            doi_and_indexes.append([alias_type, alias_value, get_index_by_alias(r,alias_type, alias_value)])
        the_valid_indexes= list(set([index for alias_type, alias_value, index in doi_and_indexes if index is not None]))
        if len(the_valid_indexes)>1:
            multirecord_doi = [[alias_type, alias_value] for alias_type, alias_value, index in doi_and_indexes if index is not None]
            conflict_doi_that_should_same_but_splited.append(multirecord_doi)
            continue
        if len(the_valid_indexes)==0:
            the_record_index = f"UA.{Prefix_Code}.{slot}"
            for alias_type, alias_value, index in doi_and_indexes:
                unique_name = format_alias(alias_type, alias_value)
                #add_alias_with_unique_name(r,alias_type, alias_value, the_record_index)
        else:
            the_record_index = the_valid_indexes[0]
            old_set = get_alias_by_index(r,the_record_index)
            #print(the_valid_indexes)
            #print(old_set)
            for alias_type, alias_value, index in doi_and_indexes:
                unique_name = format_alias(alias_type, alias_value)
                if unique_name in old_set:continue


  0%|          | 0/72000 [00:00<?, ?it/s]